In [ ]:
from dataloading import *

train_data, dev_data, test_data = load_data()

In [ ]:
import random
import numpy as np
import torch 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_DOCS = {1: 16, 2: 8, 3: 4}

train_batches = []
for n, docs in train_data.items():
    for i in range(0, len(docs), BATCH_DOCS[n]):
        train_batches.append((n, [i for i in range(i, min(i + BATCH_DOCS[n], len(docs)))]))
random.shuffle(train_batches)

dev_batches = []
for n, docs in dev_data.items():
    for i in range(0, len(docs), BATCH_DOCS[n]):
        dev_batches.append((n, [i for i in range(i, min(i + BATCH_DOCS[n], len(docs)))]))

def collate(data, n, idxs):
    docs = [data[n][i] for i in idxs]
    ids  = torch.from_numpy(np.stack([d["tokens"] for d in docs])).long()
    mask = torch.from_numpy(np.stack([d["attention_mask"] for d in docs])).long()
    y    = torch.from_numpy(np.stack([d["labels"] for d in docs]))
    return ids.to(device), mask.to(device), y.to(device)

In [ ]:
from model_arch import *

class DocClassifier(nn.Module):
    def __init__(self, base_model, hidden_dim=1024, num_labels=100):
        super().__init__()
        self.base = base_model
        self.head = nn.Linear(hidden_dim, num_labels)

    def forward(self, input_ids, attention_mask):
        B, n, L = input_ids.shape
        ids = input_ids.reshape(B * n, L)
        mask = attention_mask.reshape(B * n, L)
        h = self.base.tok_emb(ids)
        for block in self.base.blocks:
            h = block(h)
        h = self.base.ln_f(h)
        m = mask.unsqueeze(-1).to(h.dtype)
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        scores = self.head(pooled).view(B, n, -1)
        return scores.max(dim=1).values

In [ ]:
cfg = LMConfig(vocab_size=50260)
base = ThirdDecoderLM(cfg)
base.load_state_dict(torch.load("mla.pt", map_location=device))
model = DocClassifier(base).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
from sklearn.metrics import f1_score

@torch.no_grad()
def evaluate(model, data, cutoff=0.05):
    model.eval()
    P, Y = [], []
    for n, idxs in dev_batches:
        ids, mask, y = collate(data, n, idxs)
        out = model(ids, mask)
        P.append(torch.sigmoid(out).cpu())
        Y.append(y.cpu())
        
    probs, y_true = torch.cat(P).numpy(), torch.cat(Y).numpy()
    preds = (probs >= cutoff).astype(int)
    
    empty = preds.sum(1) == 0
    preds[empty, probs[empty].argmax(1)] = 1                 
    
    macro_f1 = f1_score(y_true, preds, average="macro", zero_division=0)
    return macro_f1, probs, y_true

for epoch in range(5):
    model.train()
    running_loss, count = 0.0, 0

    for step, (n, idxs) in enumerate(train_batches):
        ids, mask, y = collate(train_data, n, idxs)
        
        optimizer.zero_grad()
        out = model(ids, mask)
        loss = loss_fn(out, y.float())
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        count += 1
        
    f1, dev_probs, dev_y = evaluate(model, dev_data, cutoff=0.05)
    print("Dev Macro-F1:",  f1, "train loss:", running_loss/count)